# Predicting war events

## I. Importing essential libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
from pathlib import Path
import json

## II. Importing

In [2]:
weather = pd.read_csv("../data/all_weather_by_hour_2023-2025_v1.csv")

In [3]:
df_war_events_raw = pd.read_csv("../data/alarms-240222-010325.csv", sep=";")
df_regions = pd.read_csv("../data/regions.csv")

In [4]:
json_path_isw = Path("../data/isw_reports.json")

with open(json_path_isw, "r", encoding="utf-8") as f:
    data = json.load(f)

df_isw_raw = pd.DataFrame(data)

print(type(df_isw_raw))

<class 'pandas.core.frame.DataFrame'>


In [5]:
json_path_tg = Path("../data/telegram_data.json")

with open(json_path_tg, "r", encoding="utf-8") as f:
    data = json.load(f)

df_tg_raw = pd.DataFrame(data)

print(type(df_tg_raw))

<class 'pandas.core.frame.DataFrame'>


## III. EDA

### Weather

In [6]:
weather.head()

,city_latitude,city_longitude,city_resolvedAddress,city_address,city_timezone,city_tzoffset,day_datetime,day_datetimeEpoch,day_tempmax,day_tempmin,...,hour_pressure,hour_visibility,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions,hour_icon,hour_source,hour_stations
0,50.7469,25.3263,"Луцьк, Луцький район, Україна","Lutsk,Ukraine",Europe/Kiev,2.0,2022-02-24,1645653600,4.9,0.7,...,1020.0,0.0,91.5,0.0,NaN,0.0,Overcast,snow,obs,remote
1,50.7469,25.3263,"Луцьк, Луцький район, Україна","Lutsk,Ukraine",Europe/Kiev,2.0,2022-02-24,1645653600,4.9,0.7,...,1021.0,0.2,88.2,0.0,NaN,0.0,Partially cloudy,fog,obs,remote
2,50.7469,25.3263,"Луцьк, Луцький район, Україна","Lutsk,Ukraine",Europe/Kiev,2.0,2022-02-24,1645653600,4.9,0.7,...,1022.0,10.0,100.0,NaN,NaN,NaN,Overcast,cloudy,obs,33177099999
3,50.7469,25.3263,"Луцьк, Луцький район, Україна","Lutsk,Ukraine",Europe/Kiev,2.0,2022-02-24,1645653600,4.9,0.7,...,1021.0,0.1,92.0,0.0,NaN,0.0,Overcast,fog,obs,remote
4,50.7469,25.3263,"Луцьк, Луцький район, Україна","Lutsk,Ukraine",Europe/Kiev,2.0,2022-02-24,1645653600,4.9,0.7,...,1021.0,0.0,93.8,0.0,NaN,0.0,Overcast,cloudy,obs,remote


In [7]:
weather.shape

(608304, 65)

In [8]:
weather['hour_datetime'].head(2000)

0       00:00:00
1       01:00:00
2       02:00:00
3       03:00:00
4       04:00:00
          ...   
1995    04:00:00
1996    05:00:00
1997    06:00:00
1998    07:00:00
1999    08:00:00
Name: hour_datetime, Length: 2000, dtype: object

In [9]:
weather.columns

Index(['city_latitude', 'city_longitude', 'city_resolvedAddress',
       'city_address', 'city_timezone', 'city_tzoffset', 'day_datetime',
       'day_datetimeEpoch', 'day_tempmax', 'day_tempmin', 'day_temp',
       'day_feelslikemax', 'day_feelslikemin', 'day_feelslike', 'day_dew',
       'day_humidity', 'day_precip', 'day_precipprob', 'day_precipcover',
       'day_snow', 'day_snowdepth', 'day_windgust', 'day_windspeed',
       'day_winddir', 'day_pressure', 'day_cloudcover', 'day_visibility',
       'day_solarradiation', 'day_solarenergy', 'day_uvindex', 'day_sunrise',
       'day_sunriseEpoch', 'day_sunset', 'day_sunsetEpoch', 'day_moonphase',
       'day_conditions', 'day_description', 'day_icon', 'day_source',
       'day_preciptype', 'day_stations', 'hour_datetime', 'hour_datetimeEpoch',
       'hour_temp', 'hour_feelslike', 'hour_humidity', 'hour_dew',
       'hour_precip', 'hour_precipprob', 'hour_snow', 'hour_snowdepth',
       'hour_preciptype', 'hour_windgust', 'hour_windsp

In [10]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 608304 entries, 0 to 608303
Data columns (total 65 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   city_latitude         608304 non-null  float64
 1   city_longitude        608304 non-null  float64
 2   city_resolvedAddress  608304 non-null  object 
 3   city_address          608304 non-null  object 
 4   city_timezone         608304 non-null  object 
 5   city_tzoffset         608304 non-null  float64
 6   day_datetime          608304 non-null  object 
 7   day_datetimeEpoch     608304 non-null  int64  
 8   day_tempmax           608304 non-null  float64
 9   day_tempmin           608304 non-null  float64
 10  day_temp              608304 non-null  float64
 11  day_feelslikemax      608304 non-null  float64
 12  day_feelslikemin      608304 non-null  float64
 13  day_feelslike         608304 non-null  float64
 14  day_dew               608304 non-null  float64
 15  

In [11]:
weather.describe()

,city_latitude,city_longitude,city_tzoffset,day_datetimeEpoch,day_tempmax,day_tempmin,day_temp,day_feelslikemax,day_feelslikemin,day_feelslike,...,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex
count,608304.000000,608304.000000,608304.0,6.083040e+05,608304.000000,608304.000000,608304.000000,608304.000000,608304.000000,608304.000000,...,608304.000000,608304.000000,608304.000000,608304.000000,608304.00000,331846.000000,608304.000000,603968.000000,521042.000000,603968.000000
mean,49.143238,30.142514,2.0,1.693217e+09,15.203232,6.253993,10.721077,14.287248,4.267318,9.254504,...,0.598141,23.769943,11.399938,189.830264,1016.91840,17.010448,64.811065,142.722294,0.595017,1.413903
std,1.337209,4.303973,0.0,2.748558e+07,10.592473,8.162055,9.207622,11.577354,9.929590,10.713666,...,2.615546,11.492758,6.517607,106.383976,8.68127,9.656796,37.318628,220.920973,0.828064,2.226545
min,46.472500,22.285100,2.0,1.645654e+09,-14.300000,-50.300000,-17.500000,-21.800000,-50.300000,-27.100000,...,0.000000,0.700000,0.000000,0.000000,973.00000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,48.292400,25.935500,2.0,1.669414e+09,5.900000,-0.200000,2.800000,3.800000,-3.800000,-0.100000,...,0.000000,14.800000,7.200000,98.000000,1011.20000,10.000000,30.000000,0.000000,0.000000,0.000000
50%,49.416800,30.737100,2.0,1.693256e+09,15.100000,6.100000,10.600000,15.100000,4.200000,9.900000,...,0.000000,22.300000,10.800000,191.900000,1016.60000,15.800000,80.000000,5.600000,0.100000,0.000000
75%,50.253600,34.551700,2.0,1.717016e+09,24.600000,13.200000,18.800000,24.600000,13.200000,18.800000,...,0.000000,31.000000,15.100000,287.200000,1022.00000,24.100000,99.900000,218.300000,1.000000,2.000000
max,51.493700,37.814500,2.0,1.740780e+09,60.400000,27.200000,33.200000,60.400000,28.000000,33.900000,...,107.000000,230.400000,90.000000,360.000000,1050.00000,75.000000,100.000000,952.000000,3.400000,10.000000


In [12]:
weather.isna().sum()

city_latitude              0
city_longitude             0
city_resolvedAddress       0
city_address               0
city_timezone              0
                        ... 
hour_uvindex            4336
hour_conditions            0
hour_icon                  0
hour_source                0
hour_stations              0
Length: 65, dtype: int64

In [13]:
weather.duplicated().sum()

np.int64(0)

In [14]:
weather = weather.drop(columns=[
    "city_resolvedAddress",
    "city_address",
    "day_stations",
    "day_source",
    "hour_source",
    "hour_icon",
    "hour_stations",
    "hour_datetimeEpoch",
    'hour_preciptype', 
    "hour_visibility",
    "day_visibility",
    'day_preciptype',
    "day_icon",
    "day_description",
    "day_datetimeEpoch",
    "day_sunriseEpoch",
    "day_sunsetEpoch"
])

In [15]:
weather.head()

,city_latitude,city_longitude,city_timezone,city_tzoffset,day_datetime,day_tempmax,day_tempmin,day_temp,day_feelslikemax,day_feelslikemin,...,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions
0,50.7469,25.3263,Europe/Kiev,2.0,2022-02-24,4.9,0.7,2.6,4.0,-3.1,...,0.2,31.3,15.5,275.6,1020.0,91.5,0.0,NaN,0.0,Overcast
1,50.7469,25.3263,Europe/Kiev,2.0,2022-02-24,4.9,0.7,2.6,4.0,-3.1,...,0.2,27.7,14.8,280.3,1021.0,88.2,0.0,NaN,0.0,Partially cloudy
2,50.7469,25.3263,Europe/Kiev,2.0,2022-02-24,4.9,0.7,2.6,4.0,-3.1,...,0.1,29.2,14.4,310.0,1022.0,100.0,NaN,NaN,NaN,Overcast
3,50.7469,25.3263,Europe/Kiev,2.0,2022-02-24,4.9,0.7,2.6,4.0,-3.1,...,0.1,23.8,13.3,295.1,1021.0,92.0,0.0,NaN,0.0,Overcast
4,50.7469,25.3263,Europe/Kiev,2.0,2022-02-24,4.9,0.7,2.6,4.0,-3.1,...,0.1,24.5,13.3,305.8,1021.0,93.8,0.0,NaN,0.0,Overcast


In [16]:
weather['day_datetime'] = pd.to_datetime(weather['day_datetime'])
weather['hour_datetime'] = pd.to_datetime(weather['hour_datetime'])

weather['day'] = weather['day_datetime'].dt.day
weather['month'] = weather['day_datetime'].dt.month
weather['weekday'] = weather['day_datetime'].dt.weekday

weather['hour'] = weather['hour_datetime'].dt.hour

C:\Users\vikam\AppData\Local\Temp\ipykernel_13100\822816711.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  weather['hour_datetime'] = pd.to_datetime(weather['hour_datetime'])


In [17]:
weather['day_solarradiation'].fillna(weather['day_solarradiation'].median(), inplace=True)
weather['day_solarenergy'].fillna(weather['day_solarenergy'].median(), inplace=True)
weather['day_uvindex'].fillna(weather['day_uvindex'].median(), inplace=True)
weather['hour_solarradiation'].fillna(weather['hour_solarradiation'].median(), inplace=True)
weather['hour_solarenergy'].fillna(weather['hour_solarenergy'].median(), inplace=True)
weather['hour_uvindex'].fillna(weather['hour_uvindex'].median(), inplace=True)
weather['hour_precip'] = weather['hour_precip'].fillna(0)

C:\Users\vikam\AppData\Local\Temp\ipykernel_13100\3990248897.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  weather['day_solarradiation'].fillna(weather['day_solarradiation'].median(), inplace=True)
C:\Users\vikam\AppData\Local\Temp\ipykernel_13100\3990248897.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting

In [18]:
weather = weather.drop(columns=['day_datetime','hour_datetime'])

In [19]:
weather.isna().sum()

city_latitude          0
city_longitude         0
city_timezone          0
city_tzoffset          0
day_tempmax            0
day_tempmin            0
day_temp               0
day_feelslikemax       0
day_feelslikemin       0
day_feelslike          0
day_dew                0
day_humidity           0
day_precip             0
day_precipprob         0
day_precipcover        0
day_snow               0
day_snowdepth          0
day_windgust           0
day_windspeed          0
day_winddir            0
day_pressure           0
day_cloudcover         0
day_solarradiation     0
day_solarenergy        0
day_uvindex            0
day_sunrise            0
day_sunset             0
day_moonphase          0
day_conditions         0
hour_temp              0
hour_feelslike         0
hour_humidity          0
hour_dew               0
hour_precip            0
hour_precipprob        0
hour_snow              0
hour_snowdepth         0
hour_windgust          0
hour_windspeed         0
hour_winddir           0


In [20]:
def simplify_weather(x):
    if 'Snow' in x:
        return 'Snow'
    if 'Rain' in x:
        return 'Rain'
    if 'Overcast' in x or 'cloudy' in x:
        return 'Cloudy'
    if 'Clear' in x:
        return 'Clear'
    if 'Fog' in x:
        return 'Fog'
    return 'Other'

weather['hour_conditions_simple'] = weather['hour_conditions'].apply(simplify_weather)
weather['day_conditions_simple'] = weather['day_conditions'].apply(simplify_weather)

In [21]:
weather = pd.get_dummies(weather, columns=['day_conditions_simple','hour_conditions_simple'])

In [22]:
weather.shape

(608304, 58)

In [23]:
weather.columns

Index(['city_latitude', 'city_longitude', 'city_timezone', 'city_tzoffset',
       'day_tempmax', 'day_tempmin', 'day_temp', 'day_feelslikemax',
       'day_feelslikemin', 'day_feelslike', 'day_dew', 'day_humidity',
       'day_precip', 'day_precipprob', 'day_precipcover', 'day_snow',
       'day_snowdepth', 'day_windgust', 'day_windspeed', 'day_winddir',
       'day_pressure', 'day_cloudcover', 'day_solarradiation',
       'day_solarenergy', 'day_uvindex', 'day_sunrise', 'day_sunset',
       'day_moonphase', 'day_conditions', 'hour_temp', 'hour_feelslike',
       'hour_humidity', 'hour_dew', 'hour_precip', 'hour_precipprob',
       'hour_snow', 'hour_snowdepth', 'hour_windgust', 'hour_windspeed',
       'hour_winddir', 'hour_pressure', 'hour_cloudcover',
       'hour_solarradiation', 'hour_solarenergy', 'hour_uvindex',
       'hour_conditions', 'day', 'month', 'weekday', 'hour',
       'day_conditions_simple_Clear', 'day_conditions_simple_Cloudy',
       'day_conditions_simple_Rain', 

In [24]:
weather[['day_conditions','hour_conditions']].nunique()

day_conditions     19
hour_conditions    16
dtype: int64

In [25]:
weather = weather.drop(columns=['day_tempmax','day_tempmin', 'hour_conditions', 'day_conditions'])

In [26]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 608304 entries, 0 to 608303
Data columns (total 54 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   city_latitude                  608304 non-null  float64
 1   city_longitude                 608304 non-null  float64
 2   city_timezone                  608304 non-null  object 
 3   city_tzoffset                  608304 non-null  float64
 4   day_temp                       608304 non-null  float64
 5   day_feelslikemax               608304 non-null  float64
 6   day_feelslikemin               608304 non-null  float64
 7   day_feelslike                  608304 non-null  float64
 8   day_dew                        608304 non-null  float64
 9   day_humidity                   608304 non-null  float64
 10  day_precip                     608304 non-null  float64
 11  day_precipprob                 608304 non-null  float64
 12  day_precipcover               

In [27]:
corr = weather.corr(numeric_only=True)

In [28]:
weather.shape

(608304, 54)

In [29]:
weather.to_csv('weather_date.csv', index=False)

### War events (alarms)

In [30]:
df_war_events_raw.shape

(55788, 6)

In [31]:
df_war_events_raw.head()

,id,region_id,region_city,all_region,start,end
0,52432,12,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28
1,53292,23,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43
2,52080,3,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42
3,52857,19,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47
4,52700,18,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19


In [32]:
df_war_events_raw.sample(5)

,id,region_id,region_city,all_region,start,end
42121,143616,3,Вінницька обл.,1,2024-06-27 21:40:26,2024-06-27 22:09:13
28422,128261,19,Харківська обл.,1,2023-11-11 00:10:32,2023-11-11 00:27:25
14814,58641,5,Донецька обл.,1,2023-01-06 02:05:42,2023-01-06 02:35:29
15312,71485,24,Івано-Франківська обл.,1,2023-01-25 14:02:43,2023-01-25 15:45:00
22819,99580,17,Сумська обл.,1,2023-07-27 18:21:29,2023-07-27 18:57:58


In [33]:
df_war_events_raw.describe()

,id,region_id,all_region
count,55788.000000,55788.000000,55788.000000
mean,109103.029935,12.178121,0.972180
std,38574.559928,6.474089,0.164457
min,1.000000,1.000000,0.000000
25%,68259.750000,6.000000,1.000000
50%,126918.500000,13.000000,1.000000
75%,143399.250000,19.000000,1.000000
max,158665.000000,25.000000,1.000000


In [34]:
df_war_events_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55788 entries, 0 to 55787
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           55788 non-null  int64 
 1   region_id    55788 non-null  int64 
 2   region_city  55788 non-null  object
 3   all_region   55788 non-null  int64 
 4   start        55788 non-null  object
 5   end          55788 non-null  object
dtypes: int64(3), object(3)
memory usage: 2.6+ MB


In [35]:
print("Missing values:\n", df_war_events_raw.isna().sum())
print("\nDuplicate rows:", df_war_events_raw.duplicated().sum())
print("Duplicate id:", df_war_events_raw.duplicated(subset=["id"]).sum())

Missing values:
 id             0
region_id      0
region_city    0
all_region     0
start          0
end            0
dtype: int64

Duplicate rows: 0
Duplicate id: 0


In [36]:
df_war_events = df_war_events_raw.copy()

In [37]:
df_war_events["start"] = pd.to_datetime(df_war_events["start"], errors="coerce")
df_war_events["end"] = pd.to_datetime(df_war_events["end"], errors="coerce")

print("Invalid start dates:", df_war_events["start"].isna().sum())
print("Invalid end dates:", df_war_events["end"].isna().sum())

print(df_war_events[["start", "end"]].dtypes)

Invalid start dates: 0
Invalid end dates: 0
start    datetime64[ns]
end      datetime64[ns]
dtype: object


In [38]:
print("Min start:", df_war_events["start"].min())
print("Max start:", df_war_events["start"].max())

print("Min end:", df_war_events["end"].min())
print("Max end:", df_war_events["end"].max())

Min start: 2022-02-24 07:43:17
Max start: 2025-03-01 23:26:07
Min end: 2022-02-24 09:52:28
Max end: 2025-03-02 02:44:07


In [39]:
df_war_events["duration_min"] = (df_war_events["end"] - df_war_events["start"]).dt.total_seconds() / 60

print(df_war_events["duration_min"].describe())

count    55788.000000
mean        72.798103
std         93.094316
min       -781.700000
25%         26.566667
50%         39.733333
75%         84.716667
max       3031.300000
Name: duration_min, dtype: float64


In [40]:
print("\nNegative durations:", (df_war_events["duration_min"] < 0).sum())
print("Zero durations:", (df_war_events["duration_min"] == 0).sum())


Negative durations: 1
Zero durations: 0


In [41]:
df_war_events[df_war_events["duration_min"] < 0]

,id,region_id,region_city,all_region,start,end,duration_min
47970,150000,17,Сумська обл.,1,2024-10-01 20:53:04,2024-10-01 07:51:22,-781.7


One anomalous record with a **negative duration** was found

In [42]:
df_war_events["region_key"] = (
    df_war_events["region_city"]
    .str.replace(" обл.", "")
    .replace({"Крим": "АР Крим"})
)

In [43]:
df_war_events.head()

,id,region_id,region_city,all_region,start,end,duration_min,region_key
0,52432,12,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,129.183333,Львівська
1,53292,23,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,191.000000,Чернігівська
2,52080,3,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,30.000000,Вінницька
3,52857,19,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,48.000000,Харківська
4,52700,18,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,420.716667,Тернопільська


In [44]:
alarm_regions = sorted(
    df_war_events[df_war_events["all_region"] == 1]["region_key"].unique()
)

regions = df_regions["region"]

missing_in_alarms = sorted(set(regions) - set(alarm_regions))
extra_in_alarms = sorted(set(alarm_regions) - set(regions))

print("Missing regions in alarms:", missing_in_alarms)
print("Extra regions in alarms:", extra_in_alarms)

Missing regions in alarms: ['Луганська']
Extra regions in alarms: []


**Luhansk region is missing** in the alarms dataset.

In [45]:
alarm_id_check = (
    df_war_events[df_war_events["all_region"] == 1][["region_id", "region_key"]]
    .drop_duplicates()
    .rename(columns={"region_id": "region_id_alarms"})
)

regions_id_check = (
    df_regions[["region_id", "region"]]
    .rename(columns={"region_id": "region_id_regions", "region": "region_key"})
)

id_check = alarm_id_check.merge(regions_id_check, on="region_key")

id_mismatch = id_check[id_check["region_id_alarms"] != id_check["region_id_regions"]]

print("Number of ID mismatches:", len(id_mismatch))
id_mismatch

Number of ID mismatches: 19


,region_id_alarms,region_key,region_id_regions
0,12,Львівська,13
1,23,Чернігівська,25
2,3,Вінницька,2
3,19,Харківська,20
4,18,Тернопільська,19
5,16,Рівненська,17
6,22,Черкаська,23
7,14,Одеська,15
9,2,Волинська,3
11,20,Херсонська,21


### ISW

In [46]:
df_isw_raw.shape

(1439, 4)

In [47]:
df_isw_raw.head()

,date,title,url,text
0,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...
1,2022-02-26,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...
2,2022-02-27,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...
3,2022-02-28,"Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
4,2022-03-01,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...


In [48]:
df_isw_raw.sample(5)

,date,title,url,text
79,2022-05-21,"Russian Offensive Campaign Assessment, May 21,...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
86,2022-05-28,"Russian Offensive Campaign Assessment, May 28,...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
65,2022-05-06,"Russian Offensive Campaign Assessment, May 6, ...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
210,2022-09-30,"Russian Offensive Campaign Assessment, Septemb...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
1310,2025-10-20,"Russian Offensive Campaign Assessment, October...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...


In [49]:
df_isw_raw.describe()

,date,title,url,text
count,1439,1439,1439,1439
unique,1439,1439,1439,1439
top,2026-03-01,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...
freq,1,1,1,1


In [50]:
df_isw_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1439 entries, 0 to 1438
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   date    1439 non-null   object
 1   title   1439 non-null   object
 2   url     1439 non-null   object
 3   text    1439 non-null   object
dtypes: object(4)
memory usage: 45.1+ KB


All values are unique.  
All columns currently have the `object` data type. 

In [51]:
df_isw = df_isw_raw.copy()

In [52]:
df_isw["date"] = pd.to_datetime(df_isw["date"], errors="coerce")
print("Invalid dates:", df_isw["date"].isna().sum())
print(df_isw["date"].dtype)

Invalid dates: 0
datetime64[ns]


In [53]:
print("Duplicate dates:", df_isw["date"].duplicated().sum())

Duplicate dates: 0


In [54]:
all_dates = pd.date_range("2022-02-24", "2026-03-01")
existing_dates = pd.DatetimeIndex(df_isw["date"])
missing_dates = all_dates.difference(existing_dates)

print("Missing dates:", len(missing_dates))
print(missing_dates)

Missing dates: 28
DatetimeIndex(['2022-02-24', '2022-03-16', '2022-03-18', '2022-03-29',
               '2022-04-01', '2022-04-11', '2022-05-14', '2022-06-23',
               '2022-10-28', '2022-11-24', '2022-12-25', '2023-01-01',
               '2023-03-19', '2023-07-08', '2023-07-27', '2023-11-23',
               '2023-12-25', '2024-01-01', '2024-01-05', '2024-10-08',
               '2024-11-28', '2024-12-25', '2025-01-01', '2025-09-07',
               '2025-10-31', '2025-11-27', '2025-12-25', '2026-01-01'],
              dtype='datetime64[ns]', freq=None)


In [55]:
outside_range = df_isw[(df_isw["date"] < "2022-02-24") | (df_isw["date"] > "2026-03-01")]
print(outside_range)

Empty DataFrame
Columns: [date, title, url, text]
Index: []


In [56]:
def clean_isw_text(text):
    text = str(text)
    text = text.replace("Previous\nNext", "")
    text = text.replace("\n", " ")
    return text

df_isw["text_clean"] = df_isw["text"].apply(clean_isw_text)

In [57]:
df_isw.head()

,date,title,url,text,text_clean
0,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...,Russia-Ukraine Warning Update: Russian Offens...
1,2022-02-26,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...,Russia-Ukraine Warning Update: Russian Offens...
2,2022-02-27,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Previous\nNext\nRussia-Ukraine Warning Update:...,Russia-Ukraine Warning Update: Russian Offens...
3,2022-02-28,"Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...,"Russian Offensive Campaign Assessment, Februa..."
4,2022-03-01,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,Previous\nNext\nRussian Offensive Campaign Ass...,"Russian Offensive Campaign Assessment, March ..."


In [58]:
df_isw["text"] = df_isw["text_clean"]
df_isw = df_isw.drop(columns=["text_clean"])

In [59]:
df_isw.head()

,date,title,url,text
0,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Russia-Ukraine Warning Update: Russian Offens...
1,2022-02-26,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Russia-Ukraine Warning Update: Russian Offens...
2,2022-02-27,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Russia-Ukraine Warning Update: Russian Offens...
3,2022-02-28,"Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"Russian Offensive Campaign Assessment, Februa..."
4,2022-03-01,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,"Russian Offensive Campaign Assessment, March ..."


### Telegram 

In [60]:
df_tg_raw.shape

(129477, 3)

In [61]:
df_tg_raw.head()

,date,channel,message
0,2026-03-06 11:06:58+00:00,DeepStateUA,**🔄**** Мапу оновлено\n**\n⚔️ Ворог просунувся...
1,2026-03-06 10:57:29+00:00,DeepStateUA,**🇺🇦**** Другий день обміну: додому **[**повер...
2,2026-03-06 06:32:47+00:00,DeepStateUA,🤬 **Угорщина затримала інкасаторські автомобіл...
3,2026-03-05 14:46:58+00:00,DeepStateUA,**🔄**** Мапу оновлено\n**\n⚔️ Ворог просунувся...
4,2026-03-05 13:46:39+00:00,DeepStateUA,📋 **Зачистка з боку Сил Оборони на стику Запор...


In [62]:
df_tg_raw.sample(5)

,date,channel,message
2287,2024-11-30 21:11:01+00:00,DeepStateUA,**✍️**** Певні коригування підкладок мапи у ра...
128215,2022-07-25 07:35:36+00:00,kpszsu,⚡️Загальні бойові втрати противника з 24.02 по...
110613,2024-09-20 08:19:24+00:00,kpszsu,🚀 Полтавська область - ракетна небезпека!
48758,2023-06-02 16:47:07+00:00,UkraineNow,"Максимум уваги – ситуації на передовій, у всіх..."
46967,2023-07-21 10:21:00+00:00,UkraineNow,"**До 20 тисяч російських ув’язнених, які були ..."


In [63]:
df_tg_raw.describe()

,date,channel,message
count,129477,129477,129477
unique,129333,3,107432
top,2025-10-23 03:28:04+00:00,UkraineNow,📢 Відбій загрози.
freq,6,61363,929


In [64]:
df_tg_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129477 entries, 0 to 129476
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   date     129477 non-null  object
 1   channel  129477 non-null  object
 2   message  129477 non-null  object
dtypes: object(3)
memory usage: 3.0+ MB


In [65]:
print("Unique channels:", df_tg_raw["channel"].nunique())
print(df_tg_raw["channel"].value_counts())

Unique channels: 3
channel
UkraineNow     61363
kpszsu         56272
DeepStateUA    11842
Name: count, dtype: int64


In [66]:
print("Duplicate rows:", df_tg_raw.duplicated().sum())

Duplicate rows: 0


In [67]:
df_tg = df_tg_raw.copy()

In [68]:
df_tg["date"] = pd.to_datetime(df_tg["date"], utc=True, errors="coerce")
print("Invalid dates:", df_tg["date"].isna().sum())
print(df_isw["date"].dtype)

Invalid dates: 0
datetime64[ns]


In [69]:
print("Min date:", df_tg["date"].min())
print("Max date:", df_tg["date"].max())

Min date: 2022-02-24 03:51:12+00:00
Max date: 2026-03-06 16:49:21+00:00


In [70]:
df_tg["message"] = df_tg["message"].astype(str)

print("Empty messages:", (df_tg["message"] == "").sum())
print("Whitespace messages:", df_tg["message"].str.strip().eq("").sum())

print("\nShortest messages:")
print(df_tg.loc[df_tg["message"].str.len().nsmallest(10).index, ["channel", "date", "message"]])

Empty messages: 0
Whitespace messages: 0

Shortest messages:
           channel                      date message
1218   DeepStateUA 2025-06-01 11:49:49+00:00       🕸
4052   DeepStateUA 2024-02-16 12:20:06+00:00       🥪
7472   DeepStateUA 2022-11-11 11:10:16+00:00       🍉
9531   DeepStateUA 2022-04-12 18:46:52+00:00       😎
9804   DeepStateUA 2022-04-01 04:39:05+00:00       😳
10044  DeepStateUA 2022-03-22 16:42:05+00:00       💸
10163  DeepStateUA 2022-03-17 21:02:54+00:00       😐
10670  DeepStateUA 2022-03-04 16:47:06+00:00       🤨
10683  DeepStateUA 2022-03-04 12:53:13+00:00       🍾
10798  DeepStateUA 2022-03-03 07:02:20+00:00       🔴


## IV. Prepare data